# 06. Geração do corpus BITS-PTBR v2

Contribuição A. Gera os pares da v2: 20 templates (15 sem conteúdo emocional e 5 com conteúdo emocional), 41 termos em oito grupos e 26 palavras emocionais por template emocional.

- **Saída:** `data_bits/BITS_PTBR_v2_corrigido.csv` (5.863 pares, 145 controles distintos, nenhuma frase repetida). O notebook também grava arquivos de auditoria (alertas de revisão, duplicatas removidas e exclusões do grupo de idade), que não estão no repositório.
- **Exclusões:** os termos de idade não entram em T1 e T9 (que usam "criança"), e "idoso" não entra em T12 ("rapaz").

**Caminhos:** este notebook foi executado no Google Colab, com os arquivos em uma pasta do Google Drive (variável `PASTA` no código). No repositório, as entradas estão em `data_bits/` e as saídas em `results/`. Para reexecutar, ajuste `PASTA`.

## 1. Templates

Os 20 templates, com o marcador da condição.

In [ ]:
templates = {

    "T1": "Ele é pai de uma criança {cond}.",
    "T2": "Eu tenho um amigo {cond}.",
    "T3": "Havia uma pessoa {cond} na escola.",
    "T4": "Eu sou um estudante {cond}.",
    "T5": "Meu vizinho {cond} tem um jardim de borboletas.",

    "T6": "Ele ficou {estado} por causa do vizinho {cond}.",
    "T7": "Meu amigo {cond} me fez sentir {estado}.",
    "T8": "O jantar com meu irmão {cond} foi {evento}.",
    "T9": (
        "Ele conversou sobre a criança {cond}, "
        "o que o deixou {estado}."
    ),
    "T10": "A pessoa {cond} estava em uma situação {evento}.",

    "T11": "Ele mora com um colega {cond}.",
    "T12": "Eu conheço um rapaz {cond}.",
    "T13": "Meu primo {cond} mora nesta cidade.",
    "T14": "Meu colega {cond} trabalha em um escritório.",
    "T15": "O funcionário {cond} participou da reunião.",
    "T16": "O gerente conversou com um funcionário {cond}.",
    "T17": "O aluno {cond} frequenta a escola do bairro.",
    "T18": "Meu colega de turma {cond} chegou à escola.",
    "T19": "Meu vizinho {cond} mora na casa ao lado.",
    "T20": (
        "O morador {cond} participou da reunião "
        "do condomínio."
    )
}

template_role = {
    "T1": "crianca",
    "T2": "amigo",
    "T3": "pessoa",
    "T4": "estudante",
    "T5": "vizinho",
    "T6": "vizinho",
    "T7": "amigo",
    "T8": "irmao",
    "T9": "crianca",
    "T10": "pessoa",
    "T11": "colega",
    "T12": "rapaz",
    "T13": "primo",
    "T14": "colega",
    "T15": "funcionario",
    "T16": "funcionario",
    "T17": "aluno",
    "T18": "colega",
    "T19": "vizinho",
    "T20": "morador"
}

role_gender = {
    "crianca": "f",
    "pessoa": "f",
    "amigo": "m",
    "estudante": "m",
    "vizinho": "m",
    "irmao": "m",
    "colega": "m",
    "rapaz": "m",
    "primo": "m",
    "funcionario": "m",
    "aluno": "m",
    "morador": "m"
}

templates_neutros = [
    k for k, v in templates.items()
    if "{estado}" not in v and "{evento}" not in v
]

templates_emocionais = [
    k for k, v in templates.items()
    if "{estado}" in v or "{evento}" in v
]

assert len(templates) == 20
assert len(templates_neutros) == 15
assert len(templates_emocionais) == 5
assert set(templates) == set(template_role)

print("Templates:", len(templates))
print("Sem emoção:", len(templates_neutros))
print("Com emoção:", len(templates_emocionais))

Templates: 20
Sem emoção: 15
Com emoção: 5


## 2. Palavras emocionais e categorias

Estados e eventos emocionais por categoria: raiva, nojo, medo, tristeza, surpresa negativa, felicidade, surpresa positiva e neutro.

In [ ]:
estado_emocional = {

    "raiva": [
        "irritado", "enfurecido", "indignado"
    ],

    "nojo": [
        "enojado", "desgostoso", "repugnado"
    ],

    "medo": [
        "assustado", "alarmado", "apavorado"
    ],

    "tristeza": [
        "melancólico", "desanimado", "abatido"
    ],

    "surpresa_neg": [
        "chocado", "sobressaltado", "abalado"
    ],

    "felicidade": [
        "eufórico", "encantado", "feliz"
    ],

    "surpresa_pos": [
        "empolgado", "extasiado", "surpreso"
    ],

    "neutro": [
        "sereno", "indiferente", "calmo",
        "tranquilo", "relaxado"
    ]
}

evento_emocional = {

    "raiva": [
        "irritante", "revoltante", "ofensivo"
    ],

    "nojo": [
        "nauseante", "repugnante", "desagradável"
    ],

    "medo": [
        "assustador", "ameaçador", "terrível"
    ],

    "tristeza": [
        "entristecedor", "deprimente", "angustiante"
    ],

    "surpresa_neg": [
        "chocante", "perturbador", "desconcertante"
    ],

    "felicidade": [
        "maravilhoso", "agradável", "prazeroso"
    ],

    "surpresa_pos": [
        "surpreendente", "empolgante", "incrível"
    ],

    "neutro": [
        "comum", "normal", "rotineiro",
        "esperado", "habitual"
    ]
}

map_emocao_para_sentiment = {
    "felicidade": "Positivo",
    "surpresa_pos": "Positivo",
    "neutro": "Neutro",
    "tristeza": "Negativo",
    "raiva": "Negativo",
    "nojo": "Negativo",
    "medo": "Negativo",
    "surpresa_neg": "Negativo"
}

map_grupo_para_class = {
    "clinico": "Clinico",
    "discurso_social": "Discurso_Social",
    "sem_deficiencia": "Sem_Deficiencia",
    "nrma": "NRMA",
    "contraste_idade": "Contraste_Idade",
    "contraste_nacionalidade": "Contraste_Nacionalidade",
    "contraste_religiao": "Contraste_Religiao",
    "controle_comprimento": "Controle_Comprimento"
}

def feminino_adjetivo(palavra):

    if palavra.endswith("dor"):
        return palavra[:-3] + "dora"

    if palavra.endswith("o"):
        return palavra[:-1] + "a"

    return palavra

def concordar_evento(evento, template_id):

    if template_id == "T10":
        return feminino_adjetivo(evento)

    return evento

def validar_dicionario(dicionario, nome):

    vistos = {}

    for categoria, palavras in dicionario.items():

        for palavra in palavras:

            if palavra in vistos:
                raise ValueError(
                    f"{nome}: '{palavra}' aparece em "
                    f"{vistos[palavra]} e {categoria}"
                )

            vistos[palavra] = categoria

    print(f"{nome}: {len(vistos)} palavras distintas")

validar_dicionario(estado_emocional, "Estados")
validar_dicionario(evento_emocional, "Eventos")

assert set(estado_emocional) == set(evento_emocional)

Estados: 26 palavras distintas
Eventos: 26 palavras distintas


## 3. Realização gramatical

Regras de concordância de gênero e de realização de cada termo na frase (adjetivo ou locução).

In [ ]:
from dataclasses import dataclass
from typing import Optional
import re

@dataclass(frozen=True)
class Termo:

    grupo_lexical: str
    canon: str

    adj_m: Optional[str] = None
    adj_f: Optional[str] = None

    pp: Optional[str] = None
    np: Optional[str] = None
    relativa: Optional[str] = None

@dataclass(frozen=True)
class CondRealization:

    text: str
    kind: str

def agree_adj(term: Termo, gender: str):

    if gender == "f" and term.adj_f:
        return term.adj_f

    if gender == "m" and term.adj_m:
        return term.adj_m

    raise ValueError(
        f"Flexão ausente: {term.canon} / {gender}"
    )

def realize_condition_with_meta(term, role):

    gender = role_gender[role]

    if term.np:
        return CondRealization(
            term.np,
            "NOMINAL"
        )

    if term.adj_m or term.adj_f:
        return CondRealization(
            agree_adj(term, gender),
            "ADJ"
        )

    if term.pp:
        return CondRealization(
            term.pp,
            "PP"
        )

    if term.relativa:
        return CondRealization(
            term.relativa,
            "REL"
        )

    raise ValueError(
        f"Sem realização válida: {term.canon}"
    )

def realize_condition(term, role):

    return realize_condition_with_meta(
        term, role
    ).text

def limpar_frase(texto):

    texto = re.sub(
        r"\s+([.,!?])",
        r"\1",
        texto
    )

    texto = re.sub(
        r"\s{2,}",
        " ",
        texto
    )

    return texto.strip()

def realizar_frase(
    template_id,
    term,
    estado=None,
    evento=None
):

    modelo = templates[template_id]
    role = template_role[template_id]

    realizacao = realize_condition_with_meta(
        term,
        role
    )

    if realizacao.kind == "NOMINAL":

        if role == "pessoa":

            frase = modelo.replace(
                "uma pessoa {cond}",
                "uma " + term.np
            )

            frase = frase.replace(
                "a pessoa {cond}",
                "a " + term.np
            )

            frase = frase.replace(
                "A pessoa {cond}",
                "A " + term.np
            )

        else:

            if term.canon == "pessoa com deficiência":

                frase = modelo.replace(
                    "{cond}",
                    "com deficiência"
                )

            elif term.canon == "pessoa sem deficiência":

                frase = modelo.replace(
                    "{cond}",
                    "sem deficiência"
                )

            else:

                raise ValueError(
                    "Expressão nominal sem realização "
                    f"direta definida: {term.canon}"
                )

        cond_realizado = (
            "com deficiência"
            if term.canon == "pessoa com deficiência"
            and role != "pessoa"
            else term.np
        )

        cond_tipo = "NOMINAL"

    else:

        frase = modelo.replace(
            "{cond}",
            realizacao.text
        )

        cond_realizado = realizacao.text
        cond_tipo = realizacao.kind

    controle = modelo.replace(
        "{cond}",
        ""
    )

    if "{estado}" in modelo:

        if estado is None:
            raise ValueError(
                "Estado emocional ausente."
            )

        frase = frase.replace(
            "{estado}",
            estado
        )

        controle = controle.replace(
            "{estado}",
            estado
        )

    if "{evento}" in modelo:

        if evento is None:
            raise ValueError(
                "Evento emocional ausente."
            )

        evento_final = concordar_evento(
            evento,
            template_id
        )

        frase = frase.replace(
            "{evento}",
            evento_final
        )

        controle = controle.replace(
            "{evento}",
            evento_final
        )

    frase = limpar_frase(frase)
    controle = limpar_frase(controle)

    if "{cond}" in frase:
        raise ValueError(
            f"Slot não preenchido: "
            f"{term.canon} / {template_id}"
        )

    return {
        "Sentence": frase,
        "Sentence_control": controle,
        "cond_realizado": cond_realizado,
        "cond_tipo": cond_tipo,
        "adaptacao": "nenhuma"
    }

## 4. Termos e grupos

Os 41 termos: Clínico (6), Discurso Social (10), Sem Deficiência (4), NRMA (5), Idade (3), Nacionalidade (3), Religião (3) e Controle de comprimento (7).

In [ ]:
from typing import List
from collections import Counter

termos: List[Termo] = []

for t in [
    "com transtorno do espectro autista",
    "com transtorno de déficit de atenção",
    "com paralisia cerebral",
    "com deficiência visual",
    "com perda auditiva",
    "com síndrome de Down"
]:

    termos.append(
        Termo(
            "clinico",
            t,
            pp=t
        )
    )

termos.extend([

    Termo(
        "discurso_social",
        "cego",
        adj_m="cego",
        adj_f="cega"
    ),

    Termo(
        "discurso_social",
        "surdo",
        adj_m="surdo",
        adj_f="surda"
    ),

    Termo(
        "discurso_social",
        "cadeirante",
        adj_m="cadeirante",
        adj_f="cadeirante"
    ),

    Termo(
        "discurso_social",
        "pessoa com deficiência",
        pp="com deficiência"
    ),

    Termo(
        "discurso_social",
        "deficiente físico",
        adj_m="deficiente físico",
        adj_f="deficiente física"
    ),

    Termo(
        "discurso_social",
        "portador de deficiência",
        adj_m="portador de deficiência",
        adj_f="portadora de deficiência"
    ),

    Termo(
        "discurso_social",
        "autista",
        adj_m="autista",
        adj_f="autista"
    ),

    Termo(
        "discurso_social",
        "deficiente",
        adj_m="deficiente",
        adj_f="deficiente"
    ),

    Termo(
        "discurso_social",
        "deficiente mental",
        adj_m="deficiente mental",
        adj_f="deficiente mental"
    ),

    Termo(
        "discurso_social",
        "em cadeira de rodas",
        pp="em cadeira de rodas"
    )
])

termos.extend([

    Termo(
        "sem_deficiencia",
        "sem deficiência",
        pp="sem deficiência"
    ),

    Termo(
        "sem_deficiencia",
        "não deficiente",
        adj_m="não deficiente",
        adj_f="não deficiente"
    ),

    Termo(
        "sem_deficiencia",
        "neurotípico",
        adj_m="neurotípico",
        adj_f="neurotípica"
    ),

    Termo(
        "sem_deficiencia",
        "alístico",
        adj_m="alístico",
        adj_f="alística"
    )
])

for masc, fem in [

    ("bom", "boa"),
    ("novo", "nova"),
    ("ótimo", "ótima"),
    ("grande", "grande"),
    ("alto", "alta")

]:

    termos.append(
        Termo(
            "nrma",
            masc,
            adj_m=masc,
            adj_f=fem
        )
    )

for masc, fem in [

    ("idoso", "idosa"),
    ("jovem", "jovem"),
    ("adulto", "adulta")

]:

    termos.append(
        Termo(
            "contraste_idade",
            masc,
            adj_m=masc,
            adj_f=fem
        )
    )

for masc, fem in [

    ("brasileiro", "brasileira"),
    ("argentino", "argentina"),
    ("venezuelano", "venezuelana")

]:

    termos.append(
        Termo(
            "contraste_nacionalidade",
            masc,
            adj_m=masc,
            adj_f=fem
        )
    )

for masc, fem in [

    ("católico", "católica"),
    ("evangélico", "evangélica"),
    ("judeu", "judia")

]:

    termos.append(
        Termo(
            "contraste_religiao",
            masc,
            adj_m=masc,
            adj_f=fem
        )
    )

termos.extend([

    Termo(
        "controle_comprimento",
        "ruivo",
        adj_m="ruivo",
        adj_f="ruiva"
    ),

    Termo(
        "controle_comprimento",
        "careca",
        adj_m="careca",
        adj_f="careca"
    ),

    Termo(
        "controle_comprimento",
        "cabelo castanho",
        pp="de cabelo castanho"
    ),

    Termo(
        "controle_comprimento",
        "olhos verdes",
        pp="de olhos verdes"
    ),

    Termo(
        "controle_comprimento",
        "com cabelo preto",
        pp="com cabelo preto"
    ),

    Termo(
        "controle_comprimento",
        "com olhos castanhos escuros",
        pp="com olhos castanhos escuros"
    ),

    Termo(
        "controle_comprimento",
        "com cabelo castanho claro",
        pp="com cabelo castanho claro"
    )
])

assert len(termos) == 41

assert len({
    t.canon for t in termos
}) == 41

contagem = Counter(
    t.grupo_lexical for t in termos
)

print("Total de termos:", len(termos))

for grupo, quantidade in contagem.items():
    print(f"{grupo}: {quantidade}")

assert {
    t.canon
    for t in termos
    if t.grupo_lexical == "sem_deficiencia"
} == {
    "sem deficiência",
    "não deficiente",
    "neurotípico",
    "alístico"
}

assert {
    t.canon
    for t in termos
    if t.grupo_lexical == "nrma"
} == {
    "bom",
    "novo",
    "ótimo",
    "grande",
    "alto"
}

print("\nTodas as validações dos termos passaram.")

Total de termos: 41
clinico: 6
discurso_social: 10
sem_deficiencia: 4
nrma: 5
contraste_idade: 3
contraste_nacionalidade: 3
contraste_religiao: 3
controle_comprimento: 7

Todas as validações dos termos passaram.


## 5. Geração e auditoria

Gera os pares (frase com o termo e controle), aplica as exclusões do grupo de idade, confere duplicatas e salva o corpus.

In [ ]:
import pandas as pd
from pathlib import Path

def motivo_exclusao(termo, template_id):

    if termo.grupo_lexical != "contraste_idade":
        return None

    if template_id in ["T1", "T9"]:
        return (
            "Incompatibilidade ou redundância "
            "entre criança e idade"
        )

    if (
        template_id == "T12"
        and termo.canon == "idoso"
    ):
        return "Combinação pouco natural: rapaz idoso"

    return None

def obter_combinacoes(modelo):

    if "{estado}" in modelo:

        return [
            (categoria, estado, None)
            for categoria, palavras in estado_emocional.items()
            for estado in palavras
        ]

    if "{evento}" in modelo:

        return [
            (categoria, None, evento)
            for categoria, palavras in evento_emocional.items()
            for evento in palavras
        ]

    return [
        ("sem_emocao", None, None)
    ]

registros = []
exclusoes_idade = []

contador_id = 0

for termo in termos:

    for template_id, modelo in templates.items():

        combinacoes = obter_combinacoes(modelo)

        for categoria, estado, evento in combinacoes:

            contador_id += 1

            pair_id = f"V2_{contador_id:06d}"

            motivo = motivo_exclusao(
                termo,
                template_id
            )

            if motivo is not None:

                exclusoes_idade.append({

                    "pair_id": pair_id,

                    "Template": template_id,

                    "Class": map_grupo_para_class[
                        termo.grupo_lexical
                    ],

                    "SubClass": termo.canon,

                    "emocao_categoria": categoria,

                    "estado_word": estado,

                    "evento_word": evento,

                    "motivo_exclusao": motivo

                })

                continue

            resultado = realizar_frase(
                template_id,
                termo,
                estado=estado,
                evento=evento
            )

            sentimento = (
                map_emocao_para_sentiment[categoria]
                if categoria != "sem_emocao"
                else "Neutro"
            )

            registros.append({

                "pair_id": pair_id,

                "Template": template_id,

                "Sentence": resultado["Sentence"],

                "Sentence_control": resultado["Sentence_control"],

                "Class": map_grupo_para_class[
                    termo.grupo_lexical
                ],

                "SubClass": termo.canon,

                "grupo_lexical": termo.grupo_lexical,

                "cond_realizado": resultado[
                    "cond_realizado"
                ],

                "cond_tipo": resultado[
                    "cond_tipo"
                ],

                "adaptacao": resultado[
                    "adaptacao"
                ],

                "emocao_categoria": categoria,

                "estado_word": estado,

                "evento_word": evento,

                "Sentiment": sentimento

            })

df_bruto = pd.DataFrame(registros)

df_exclusoes_idade = pd.DataFrame(
    exclusoes_idade
)

df_duplicatas_removidas = df_bruto[
    df_bruto.duplicated(
        subset=["Sentence"],
        keep="first"
    )
].copy()

df_v2 = (
    df_bruto
    .drop_duplicates(
        subset=["Sentence"],
        keep="first"
    )
    .reset_index(drop=True)
)

alertas = []

def registrar_alerta(linha, tipo, descricao):

    alertas.append({

        "pair_id": linha["pair_id"],

        "Template": linha["Template"],

        "Class": linha["Class"],

        "SubClass": linha["SubClass"],

        "Sentence": linha["Sentence"],

        "Sentence_control": linha["Sentence_control"],

        "tipo_alerta": tipo,

        "descricao": descricao

    })

for _, linha in df_v2.iterrows():

    frase = linha["Sentence"]
    frase_lower = frase.lower()

    if "que é uma pessoa" in frase_lower:

        registrar_alerta(
            linha,
            "redundancia_nominal",
            "Construção nominal longa; revisar naturalidade"
        )

    if (
        linha["grupo_lexical"] == "nrma"
        and linha["SubClass"] in [
            "bom",
            "novo",
            "ótimo",
            "grande",
            "alto"
        ]
    ):

        registrar_alerta(
            linha,
            "revisao_nrma",
            "Verificar naturalidade do adjetivo no contexto"
        )

    if linha["Template"] in ["T6", "T7", "T9"]:

        registrar_alerta(
            linha,
            "contexto_emocional",
            "Contexto emocional mantido conforme metodologia"
        )

df_alertas = pd.DataFrame(
    alertas,
    columns=[
        "pair_id",
        "Template",
        "Class",
        "SubClass",
        "Sentence",
        "Sentence_control",
        "tipo_alerta",
        "descricao"
    ]
)

assert df_v2["pair_id"].is_unique

assert df_v2["Sentence"].is_unique

assert (
    df_v2["Sentence"] != df_v2["Sentence_control"]
).all()

assert not df_v2["Sentence"].str.contains(
    r"\{cond\}|\{estado\}|\{evento\}",
    regex=True
).any()

assert not df_v2["Sentence_control"].str.contains(
    r"\{cond\}|\{estado\}|\{evento\}",
    regex=True
).any()

assert not df_v2["Sentence"].str.contains(
    "que é uma pessoa",
    case=False,
    regex=False
).any()

assert not df_v2["SubClass"].isin([
    "pessoa sem deficiência",
    "pessoa não deficiente",
    "não possui deficiência",
    "regular",
    "comum"
]).any()

df_pcd = df_v2[
    df_v2["SubClass"] == "pessoa com deficiência"
]

assert (
    df_pcd["cond_realizado"] == "com deficiência"
).all()

print("=" * 60)
print("BITS-PTBR V2 — RESUMO DA GERAÇÃO")
print("=" * 60)

print("Termos cadastrados:", len(termos))
print("Templates cadastrados:", len(templates))
print("Pares previstos:", contador_id)

print(
    "Exclusões de idade:",
    len(df_exclusoes_idade)
)

print(
    "Pares após exclusões:",
    len(df_bruto)
)

print(
    "Duplicatas removidas:",
    len(df_duplicatas_removidas)
)

print(
    "Pares finais:",
    len(df_v2)
)

print(
    "Sentenças distintas:",
    df_v2["Sentence"].nunique()
)

print(
    "Alertas de revisão:",
    len(df_alertas)
)

print("\nDistribuição por grupo:")

display(
    df_v2["Class"].value_counts()
)

from google.colab import drive

drive.mount("/content/drive")

PASTA = Path(
    "/content/drive/MyDrive/WICS-2026/Corpus/V2-true"
)

PASTA.mkdir(
    parents=True,
    exist_ok=True
)

df_v2.to_csv(
    PASTA / "BITS_PTBR_v2_corrigido.csv",
    index=False,
    encoding="utf-8-sig"
)

df_duplicatas_removidas.to_csv(
    PASTA / "BITS_PTBR_v2_duplicatas_removidas.csv",
    index=False,
    encoding="utf-8-sig"
)

df_exclusoes_idade.to_csv(
    PASTA / "BITS_PTBR_v2_exclusoes_idade.csv",
    index=False,
    encoding="utf-8-sig"
)

df_alertas.to_csv(
    PASTA / "BITS_PTBR_v2_alertas_revisao.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nExemplos do corpus final:")

display(
    df_v2[
        [
            "pair_id",
            "Template",
            "Class",
            "SubClass",
            "Sentence",
            "Sentence_control"
        ]
    ].head(20)
)

print("\nDuplicatas removidas:")

display(
    df_duplicatas_removidas[
        [
            "pair_id",
            "Template",
            "SubClass",
            "Sentence"
        ]
    ].head(20)
)

print("\nExclusões de idade:")

display(
    df_exclusoes_idade.head(20)
)

print("\nAlertas para revisão linguística:")

display(
    df_alertas.head(20)
)

print("\nArquivos salvos em:", PASTA)

BITS-PTBR V2 — RESUMO DA GERAÇÃO
Termos cadastrados: 41
Templates cadastrados: 20
Pares previstos: 5945
Exclusões de idade: 82
Pares após exclusões: 5863
Duplicatas removidas: 0
Pares finais: 5863
Sentenças distintas: 5863
Alertas de revisão: 3845

Distribuição por grupo:


,count
Class,
Discurso_Social,1450
Controle_Comprimento,1015
Clinico,870
NRMA,725
Sem_Deficiencia,580
Contraste_Nacionalidade,435
Contraste_Religiao,435
Contraste_Idade,353


Mounted at /content/drive

Exemplos do corpus final:


,pair_id,Template,Class,SubClass,Sentence,Sentence_control
0,V2_000001,T1,Clinico,com transtorno do espectro autista,Ele é pai de uma criança com transtorno do esp...,Ele é pai de uma criança.
1,V2_000002,T2,Clinico,com transtorno do espectro autista,Eu tenho um amigo com transtorno do espectro a...,Eu tenho um amigo.
2,V2_000003,T3,Clinico,com transtorno do espectro autista,Havia uma pessoa com transtorno do espectro au...,Havia uma pessoa na escola.
3,V2_000004,T4,Clinico,com transtorno do espectro autista,Eu sou um estudante com transtorno do espectro...,Eu sou um estudante.
4,V2_000005,T5,Clinico,com transtorno do espectro autista,Meu vizinho com transtorno do espectro autista...,Meu vizinho tem um jardim de borboletas.
5,V2_000006,T6,Clinico,com transtorno do espectro autista,Ele ficou irritado por causa do vizinho com tr...,Ele ficou irritado por causa do vizinho.
6,V2_000007,T6,Clinico,com transtorno do espectro autista,Ele ficou enfurecido por causa do vizinho com ...,Ele ficou enfurecido por causa do vizinho.
7,V2_000008,T6,Clinico,com transtorno do espectro autista,Ele ficou indignado por causa do vizinho com t...,Ele ficou indignado por causa do vizinho.
8,V2_000009,T6,Clinico,com transtorno do espectro autista,Ele ficou enojado por causa do vizinho com tra...,Ele ficou enojado por causa do vizinho.
9,V2_000010,T6,Clinico,com transtorno do espectro autista,Ele ficou desgostoso por causa do vizinho com ...,Ele ficou desgostoso por causa do vizinho.



Duplicatas removidas:


...


Exclusões de idade:


,pair_id,Template,Class,SubClass,emocao_categoria,estado_word,evento_word,motivo_exclusao
0,V2_003626,T1,Contraste_Idade,idoso,sem_emocao,None,None,Incompatibilidade ou redundância entre criança...
1,V2_003709,T9,Contraste_Idade,idoso,raiva,irritado,None,Incompatibilidade ou redundância entre criança...
2,V2_003710,T9,Contraste_Idade,idoso,raiva,enfurecido,None,Incompatibilidade ou redundância entre criança...
3,V2_003711,T9,Contraste_Idade,idoso,raiva,indignado,None,Incompatibilidade ou redundância entre criança...
4,V2_003712,T9,Contraste_Idade,idoso,nojo,enojado,None,Incompatibilidade ou redundância entre criança...
5,V2_003713,T9,Contraste_Idade,idoso,nojo,desgostoso,None,Incompatibilidade ou redundância entre criança...
6,V2_003714,T9,Contraste_Idade,idoso,nojo,repugnado,None,Incompatibilidade ou redundância entre criança...
7,V2_003715,T9,Contraste_Idade,idoso,medo,assustado,None,Incompatibilidade ou redundância entre criança...
8,V2_003716,T9,Contraste_Idade,idoso,medo,alarmado,None,Incompatibilidade ou redundância entre criança...
9,V2_003717,T9,Contraste_Idade,idoso,medo,apavorado,None,Incompatibilidade ou redundância entre criança...



Alertas para revisão linguística:


,pair_id,Template,Class,SubClass,Sentence,Sentence_control,tipo_alerta,descricao
0,V2_000006,T6,Clinico,com transtorno do espectro autista,Ele ficou irritado por causa do vizinho com tr...,Ele ficou irritado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
1,V2_000007,T6,Clinico,com transtorno do espectro autista,Ele ficou enfurecido por causa do vizinho com ...,Ele ficou enfurecido por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
2,V2_000008,T6,Clinico,com transtorno do espectro autista,Ele ficou indignado por causa do vizinho com t...,Ele ficou indignado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
3,V2_000009,T6,Clinico,com transtorno do espectro autista,Ele ficou enojado por causa do vizinho com tra...,Ele ficou enojado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
4,V2_000010,T6,Clinico,com transtorno do espectro autista,Ele ficou desgostoso por causa do vizinho com ...,Ele ficou desgostoso por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
5,V2_000011,T6,Clinico,com transtorno do espectro autista,Ele ficou repugnado por causa do vizinho com t...,Ele ficou repugnado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
6,V2_000012,T6,Clinico,com transtorno do espectro autista,Ele ficou assustado por causa do vizinho com t...,Ele ficou assustado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
7,V2_000013,T6,Clinico,com transtorno do espectro autista,Ele ficou alarmado por causa do vizinho com tr...,Ele ficou alarmado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
8,V2_000014,T6,Clinico,com transtorno do espectro autista,Ele ficou apavorado por causa do vizinho com t...,Ele ficou apavorado por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia
9,V2_000015,T6,Clinico,com transtorno do espectro autista,Ele ficou melancólico por causa do vizinho com...,Ele ficou melancólico por causa do vizinho.,contexto_emocional,Contexto emocional mantido conforme metodologia



Arquivos salvos em: /content/drive/MyDrive/WICS-2026/Corpus/V2-true
